In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
import optuna
import warnings
warnings.filterwarnings('ignore')

pd.set_option("display.max_columns", None)
print("library import is completed")

# ==========================================
# 1. データの読み込み
# ==========================================
path = "/kaggle/input/competitions/playground-series-s6e3/"
train_df = pd.read_csv(path + "train.csv")
test_df = pd.read_csv(path + "test.csv")

# ==========================================
# 2. 特徴量エンジニアリング (前回と同等)
# ==========================================
def advanced_feature_engineering(df):
    df = df.copy()
    df['Is_Family'] = ((df['Partner'] == 'Yes') | (df['Dependents'] == 'Yes')).astype(int)
    df['Senior_Single'] = ((df['SeniorCitizen'] == 1) & (df['Is_Family'] == 0)).astype(int)
    df['Is_New_Customer'] = (df['tenure'] <= 3).astype(int)
    df['Tenure_Years'] = df['tenure'] / 12.0
    
    security_services = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport']
    streaming_services = ['StreamingTV', 'StreamingMovies']
    df['Count_Security'] = df[security_services].apply(lambda x: (x == 'Yes').sum(), axis=1)
    df['Count_Streaming'] = df[streaming_services].apply(lambda x: (x == 'Yes').sum(), axis=1)
    df['Count_Total_Services'] = df[security_services + streaming_services].apply(lambda x: (x == 'Yes').sum(), axis=1)
    
    df['Has_Internet'] = (df['InternetService'] != 'No').astype(int)
    df['Is_Auto_Payment'] = df['PaymentMethod'].str.contains('automatic').astype(int)
    df['Is_High_Risk_Combo'] = ((df['Contract'] == 'Month-to-month') & (df['PaymentMethod'] == 'Electronic check')).astype(int)
    
    df['Avg_Monthly_Charges'] = df['TotalCharges'] / (df['tenure'] + 1e-5)
    df['Charges_Diff_Current_Avg'] = df['MonthlyCharges'] - df['Avg_Monthly_Charges']
    df['Expected_Total_Charges'] = df['MonthlyCharges'] * df['tenure']
    df['Total_Charges_Diff'] = df['TotalCharges'] - df['Expected_Total_Charges']
    df['Total_Charges_Ratio'] = df['TotalCharges'] / (df['Expected_Total_Charges'] + 1e-5)
    
    df['Contract_Payment_Cross'] = df['Contract'] + "_" + df['PaymentMethod']
    return df

train_df = advanced_feature_engineering(train_df)
test_df = advanced_feature_engineering(test_df)

train_df["Churn"] = train_df["Churn"].map({'Yes': 1, 'No': 0})
X = train_df.drop(columns=["id", "Churn"])
y = train_df["Churn"]
X_test = test_df.drop(columns=["id", "Churn"], errors='ignore')

# 全モデルで扱えるよう、カテゴリ変数をLabel Encoding（数値化）する
cat_features = X.select_dtypes(include=['object', 'category']).columns.tolist()
for col in cat_features:
    le = LabelEncoder()
    # 欠損値対策として文字列に変換してからエンコード
    X[col] = le.fit_transform(X[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))

# ==========================================
# 3. Optunaによる各モデルの軽めのハイパラチューニング
# ==========================================
print("Starting Optuna Hyperparameter Tuning (Light Mode)...")
optuna.logging.set_verbosity(optuna.logging.WARNING) # ログを見やすくする

# チューニング評価用のCV
tune_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

def tune_lgbm(trial):
    params = {
        'objective': 'binary', 'metric': 'auc', 'random_state': 42, 'n_jobs': -1,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'num_leaves': trial.suggest_int('num_leaves', 15, 63),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.9),
        'subsample': trial.suggest_float('subsample', 0.5, 0.9),
        'n_estimators': 300 # 軽めにするため固定
    }
    model = lgb.LGBMClassifier(**params)
    score = cross_val_score(model, X, y, cv=tune_cv, scoring='roc_auc').mean()
    return score

def tune_xgb(trial):
    params = {
        'eval_metric': 'auc', 'random_state': 42, 'n_jobs': -1,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.9),
        'subsample': trial.suggest_float('subsample', 0.5, 0.9),
        'n_estimators': 300
    }
    model = xgb.XGBClassifier(**params)
    score = cross_val_score(model, X, y, cv=tune_cv, scoring='roc_auc').mean()
    return score

# ※ 時間がかかりすぎるのを防ぐため、各モデル5〜10回の試行（n_trials）に留めます
study_lgb = optuna.create_study(direction='maximize')
study_lgb.optimize(tune_lgbm, n_trials=10)
print(f"LGBM Best AUC: {study_lgb.best_value:.5f}")

study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(tune_xgb, n_trials=10)
print(f"XGB Best AUC: {study_xgb.best_value:.5f}")

# ==========================================
# 4. スタッキング（Level 1: ベースモデルの学習とOOF予測）
# ==========================================
print("\nStarting Level 1: Base Models Training & OOF Generation...")

n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

# Level 1の予測結果（OOF）を格納する配列
oof_lgb = np.zeros(len(X))
oof_xgb = np.zeros(len(X))
oof_cb  = np.zeros(len(X))
oof_hgb = np.zeros(len(X))
oof_rf  = np.zeros(len(X))

# テストデータの予測結果格納配列
test_lgb = np.zeros(len(X_test))
test_xgb = np.zeros(len(X_test))
test_cb  = np.zeros(len(X_test))
test_hgb = np.zeros(len(X_test))
test_rf  = np.zeros(len(X_test))

# モデルの定義（Optunaのベストパラメータ + 固定パラメータモデル）
lgb_params = study_lgb.best_params
lgb_params.update({'objective': 'binary', 'random_state': 42, 'n_estimators': 1000})

xgb_params = study_xgb.best_params
xgb_params.update({'eval_metric': 'auc', 'random_state': 42, 'n_estimators': 1000})

# 以下は計算コストを抑えるため固定パラメータで設定
cb_params = {'iterations': 1000, 'learning_rate': 0.05, 'depth': 6, 'eval_metric': 'AUC', 'random_seed': 42, 'verbose': False}
hgb_params = {'learning_rate': 0.05, 'max_iter': 500, 'max_depth': 6, 'random_state': 42}
rf_params = {'n_estimators': 500, 'max_depth': 8, 'n_jobs': -1, 'random_state': 42}

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"--- Fold {fold+1} ---")
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    # 1. LightGBM
    m_lgb = lgb.LGBMClassifier(**lgb_params)
    m_lgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(50, verbose=False)])
    oof_lgb[val_idx] = m_lgb.predict_proba(X_val)[:, 1]
    test_lgb += m_lgb.predict_proba(X_test)[:, 1] / n_splits
    
    # 2. XGBoost
    m_xgb = xgb.XGBClassifier(**xgb_params, early_stopping_rounds=50)
    m_xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    oof_xgb[val_idx] = m_xgb.predict_proba(X_val)[:, 1]
    test_xgb += m_xgb.predict_proba(X_test)[:, 1] / n_splits
    
    # 3. CatBoost
    m_cb = cb.CatBoostClassifier(**cb_params, early_stopping_rounds=50)
    m_cb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    oof_cb[val_idx] = m_cb.predict_proba(X_val)[:, 1]
    test_cb += m_cb.predict_proba(X_test)[:, 1] / n_splits
    
    # 4. HistGradientBoosting (Early stopping内蔵対応)
    m_hgb = HistGradientBoostingClassifier(**hgb_params, early_stopping=True, validation_fraction=0.1)
    m_hgb.fit(X_tr, y_tr)
    oof_hgb[val_idx] = m_hgb.predict_proba(X_val)[:, 1]
    test_hgb += m_hgb.predict_proba(X_test)[:, 1] / n_splits
    
    # 5. RandomForest
    m_rf = RandomForestClassifier(**rf_params)
    m_rf.fit(X_tr, y_tr)
    oof_rf[val_idx] = m_rf.predict_proba(X_val)[:, 1]
    test_rf += m_rf.predict_proba(X_test)[:, 1] / n_splits

print("\nLevel 1 OOF AUC Scores:")
print(f"LightGBM : {roc_auc_score(y, oof_lgb):.5f}")
print(f"XGBoost  : {roc_auc_score(y, oof_xgb):.5f}")
print(f"CatBoost : {roc_auc_score(y, oof_cb):.5f}")
print(f"HistGB   : {roc_auc_score(y, oof_hgb):.5f}")
print(f"RandomF  : {roc_auc_score(y, oof_rf):.5f}")

# ==========================================
# 5. スタッキング（Level 2: メタモデルによる最終予測）
# ==========================================
print("\nStarting Level 2: Meta Model Training...")

# 1層目の予測値を特徴量とするデータフレームを作成
X_oof = pd.DataFrame({
    'lgb': oof_lgb, 'xgb': oof_xgb, 'cb': oof_cb, 'hgb': oof_hgb, 'rf': oof_rf
})
X_test_meta = pd.DataFrame({
    'lgb': test_lgb, 'xgb': test_xgb, 'cb': test_cb, 'hgb': test_hgb, 'rf': test_rf
})

# メタモデルには過学習しにくいロジスティック回帰を使用
meta_model = LogisticRegression(random_state=42)

oof_meta = np.zeros(len(X_oof))
test_meta = np.zeros(len(X_test_meta))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_oof, y)):
    X_tr_meta, y_tr_meta = X_oof.iloc[train_idx], y.iloc[train_idx]
    X_val_meta, y_val_meta = X_oof.iloc[val_idx], y.iloc[val_idx]
    
    meta_model.fit(X_tr_meta, y_tr_meta)
    oof_meta[val_idx] = meta_model.predict_proba(X_val_meta)[:, 1]
    test_meta += meta_model.predict_proba(X_test_meta)[:, 1] / n_splits

final_auc = roc_auc_score(y, oof_meta)
print(f"\n🚀 Final Stacking CV AUC: {final_auc:.5f}")

# ==========================================
# 6. 提出用ファイルの作成
# ==========================================
sub_df = pd.DataFrame({
    "id": test_df["id"],
    "Churn": test_meta
})

name = "submission_Stacking_Optuna.csv"
sub_df.to_csv(name, index=False)
print(f"{name}の出力が完了しました")

library import is completed
Data import is completed
Start Training with Advanced Features...
[LightGBM] [Info] Number of positive: 107054, number of negative: 368301
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.033996 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2029
[LightGBM] [Info] Number of data points in the train set: 475355, number of used features: 35
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.225209 -> initscore=-1.235567
[LightGBM] [Info] Start training from score -1.235567
[500]	valid_0's auc: 0.915636
[1000]	valid_0's auc: 0.915932
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Fold 1 AUC: 0.91596
[LightGBM] [Info] Number of positive: 107054, number of negative: 368301
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.032107 seconds.
You can set 